In [ ]:
import sqlite3
import pandas as pd
import numpy as np

db_path = r"D:\gbm_project\chembl\chembl_37\chembl_37_sqlite\chembl_37.db"
con = sqlite3.connect("file:" + db_path.replace("\\", "/") + "?mode=ro", uri=True)
cur = con.cursor()

cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
tables = [row[0] for row in cur.fetchall()]

print(len(tables), "جدول")
wanted = ["activities", "assays", "target_dictionary", "molecule_dictionary", "compound_structures"]
for name in wanted:
    print(name, "->", name in tables)

72 جدول
activities -> True
assays -> True
target_dictionary -> True
molecule_dictionary -> True
compound_structures -> True


In [4]:
for name in ["target_dictionary", "assays", "activities"]:
    cur.execute(f"PRAGMA table_info({name})")
    cols = [row[1] for row in cur.fetchall()]
    print(name, "->", cols)
    print()

target_dictionary -> ['tid', 'target_type', 'pref_name', 'tax_id', 'organism', 'chembl_id', 'species_group_flag']

assays -> ['assay_id', 'doc_id', 'description', 'assay_type', 'assay_test_type', 'assay_category', 'assay_organism', 'assay_tax_id', 'assay_strain', 'assay_tissue', 'assay_cell_type', 'assay_subcellular_fraction', 'tid', 'relationship_type', 'confidence_score', 'src_id', 'src_assay_id', 'chembl_id', 'cell_id', 'bao_format', 'tissue_id', 'variant_id', 'aidx', 'assay_group']

activities -> ['activity_id', 'assay_id', 'doc_id', 'record_id', 'molregno', 'standard_relation', 'standard_value', 'standard_units', 'standard_flag', 'standard_type', 'activity_comment', 'data_validity_comment', 'potential_duplicate', 'pchembl_value', 'bao_endpoint', 'uo_units', 'qudt_units', 'toid', 'upper_value', 'standard_upper_value', 'src_id', 'type', 'relation', 'value', 'units', 'text_value', 'standard_text_value', 'action_type', 'modality']



In [5]:
sql = """
SELECT COUNT(*)
FROM activities AS a
JOIN assays AS s ON a.assay_id = s.assay_id
JOIN target_dictionary AS t ON s.tid = t.tid
WHERE t.chembl_id = ?
  AND a.standard_type IN ('IC50', 'Ki', 'Kd')
  AND a.standard_relation = '='
  AND a.standard_units = 'nM'
  AND s.assay_type = 'B'
"""
cur.execute(sql, ("CHEMBL5023",))
print(cur.fetchone()[0])

6340


In [6]:
for name in ["molecule_dictionary", "compound_structures"]:
    cur.execute(f"PRAGMA table_info({name})")
    cols = [row[1] for row in cur.fetchall()]
    print(name, "->", cols)
    print()

molecule_dictionary -> ['molregno', 'pref_name', 'chembl_id', 'max_phase', 'therapeutic_flag', 'dosed_ingredient', 'structure_type', 'molecule_type', 'first_approval', 'oral', 'parenteral', 'topical', 'black_box_warning', 'natural_product', 'first_in_class', 'chirality', 'prodrug', 'inorganic_flag', 'usan_year', 'availability_type', 'usan_stem', 'polymer_flag', 'usan_substem', 'usan_stem_definition', 'withdrawn_flag', 'chemical_probe', 'orphan', 'veterinary']

compound_structures -> ['molregno', 'molfile', 'standard_inchi', 'standard_inchi_key', 'canonical_smiles']



In [7]:
import pandas as pd

sql = """
SELECT
    a.activity_id,
    m.chembl_id AS molecule_chembl_id,
    cs.canonical_smiles,
    a.standard_type,
    a.standard_value,
    a.standard_units,
    a.standard_relation,
    s.chembl_id AS assay_chembl_id,
    s.assay_type,
    t.chembl_id AS target_chembl_id,
    a.pchembl_value,
    a.data_validity_comment,
    a.potential_duplicate
FROM activities AS a
JOIN assays AS s ON a.assay_id = s.assay_id
JOIN target_dictionary AS t ON s.tid = t.tid
JOIN molecule_dictionary AS m ON a.molregno = m.molregno
LEFT JOIN compound_structures AS cs ON a.molregno = cs.molregno
WHERE t.chembl_id = ?
  AND a.standard_type IN ('IC50', 'Ki', 'Kd')
  AND a.standard_relation = '='
  AND a.standard_units = 'nM'
  AND s.assay_type = 'B'
"""

raw_mdm2 = pd.read_sql_query(sql, con, params=("CHEMBL5023",))
print(raw_mdm2.shape)
print(raw_mdm2.head())
raw_mdm2.to_csv("chembl37_mdm2_raw.csv", index=False)

(6340, 13)
   activity_id molecule_chembl_id  \
0      1427446       CHEMBL178578   
1      1427447       CHEMBL361103   
2      1427448       CHEMBL182051   
3      1427449       CHEMBL360540   
4      1427450       CHEMBL427316   

                                    canonical_smiles standard_type  \
0  O=C(O)[C@H](c1ccccc1)N1C(=O)c2cc(I)ccc2NC(=O)[...          IC50   
1  O=C(O)[C@H](c1ccc(Cl)cc1)N1C(=O)c2cc(I)ccc2NC(...          IC50   
2  C[C@H](c1ccc(Cl)cc1)N(C(=O)/C=C/c1ccccc1)[C@H]...          IC50   
3  C[C@H](c1ccc(Cl)cc1)N1C(=O)C=C(c2ccccc2)N(CCCC...          IC50   
4  C[C@H](c1ccc(Cl)cc1)N1C(=O)C=C(c2ccccc2Br)N(CC...          IC50   

   standard_value standard_units standard_relation assay_chembl_id assay_type  \
0          1700.0             nM                 =    CHEMBL874553          B   
1           200.0             nM                 =    CHEMBL874553          B   
2         30000.0             nM                 =    CHEMBL874553          B   
3         13000.0    

In [8]:
print(raw_mdm2["standard_type"].value_counts())
print()
print("مولکول یکتا:", raw_mdm2["molecule_chembl_id"].nunique())
print("بدون SMILES:", raw_mdm2["canonical_smiles"].isna().sum())
print("standard_value <= 0:", (raw_mdm2["standard_value"] <= 0).sum())
print()
print(raw_mdm2["data_validity_comment"].value_counts(dropna=False))
print()
print(raw_mdm2["potential_duplicate"].value_counts())

standard_type
IC50    5379
Ki       749
Kd       212
Name: count, dtype: int64

مولکول یکتا: 4807
بدون SMILES: 4
standard_value <= 0: 0

data_validity_comment
None                             6304
Outside typical range              33
Potential transcription error       3
Name: count, dtype: int64

potential_duplicate
0    5910
1     430
Name: count, dtype: int64


In [9]:
clean = raw_mdm2.copy()
print("شروع:", len(clean))

clean = clean.dropna(subset=["canonical_smiles"])
print("بعد از حذف بدون SMILES:", len(clean))

clean = clean[clean["data_validity_comment"].isna()]
print("بعد از حذف هشدار ChEMBL:", len(clean))

clean = clean[clean["potential_duplicate"] == 0]
print("بعد از حذف تکراری‌ها:", len(clean))

print("مولکول یکتا:", clean["molecule_chembl_id"].nunique())
print(clean["standard_type"].value_counts())

شروع: 6340
بعد از حذف بدون SMILES: 6336
بعد از حذف هشدار ChEMBL: 6300
بعد از حذف تکراری‌ها: 5878
مولکول یکتا: 4785
standard_type
IC50    5007
Ki       690
Kd       181
Name: count, dtype: int64


In [10]:
import numpy as np

clean = clean.copy()
clean["pActivity"] = 9 - np.log10(clean["standard_value"])

print(clean["pActivity"].describe())
print()
print("pchembl خالی:", clean["pchembl_value"].isna().sum())
diff = (clean["pActivity"] - clean["pchembl_value"]).abs()
print("بیشترین اختلاف با pchembl:", diff.max())

count    5878.000000
mean        7.547755
std         1.386827
min         4.000000
25%         6.607303
50%         7.790485
75%         8.549905
max        10.443697
Name: pActivity, dtype: float64

pchembl خالی: 0
بیشترین اختلاف با pchembl: 0.005498770613082016


In [11]:
mol_mdm2 = (
    clean.groupby("molecule_chembl_id")
         .agg(
             canonical_smiles=("canonical_smiles", "first"),
             pActivity_median=("pActivity", "median"),
             pActivity_min=("pActivity", "min"),
             pActivity_max=("pActivity", "max"),
             n_measurements=("pActivity", "size"),
         )
         .reset_index()
)

mol_mdm2["spread"] = mol_mdm2["pActivity_max"] - mol_mdm2["pActivity_min"]

print(mol_mdm2.shape)
print(mol_mdm2["n_measurements"].value_counts().head())
print("مولکول با پراکندگی > 1:", (mol_mdm2["spread"] > 1).sum())
mol_mdm2.to_csv("chembl37_mdm2_molecules.csv", index=False)

(4785, 7)
n_measurements
1    3947
2     680
3     111
4      32
5       6
Name: count, dtype: int64
مولکول با پراکندگی > 1: 233


In [12]:
actives = mol_mdm2[mol_mdm2["pActivity_median"] >= 7]
print("active:", len(actives), "از", len(mol_mdm2))
print("  با فقط ۱ اندازه‌گیری:", (actives["n_measurements"] == 1).sum())
print("  با پراکندگی > 1:", (actives["spread"] > 1).sum())

active: 3240 از 4785
  با فقط ۱ اندازه‌گیری: 2625
  با پراکندگی > 1: 174


In [13]:
names_sql = """
SELECT chembl_id, pref_name, max_phase
FROM molecule_dictionary
WHERE UPPER(pref_name) LIKE '%NUTLIN%'
   OR UPPER(pref_name) LIKE '%NAVTEMADLIN%'
   OR UPPER(pref_name) LIKE '%MILADEMETAN%'
"""
known = pd.read_sql_query(names_sql, con)
known = known.rename(columns={"chembl_id": "molecule_chembl_id"})

check = known.merge(mol_mdm2, on="molecule_chembl_id", how="left")
print(check[["molecule_chembl_id", "pref_name", "max_phase",
             "pActivity_median", "n_measurements"]])

  molecule_chembl_id             pref_name  max_phase  pActivity_median  \
0       CHEMBL191334              NUTLIN-3        NaN          7.045757   
1       CHEMBL407632              NUTLIN-2        NaN          6.853872   
2      CHEMBL2402737           IDASANUTLIN        3.0          8.221849   
3      CHEMBL3125702           NAVTEMADLIN        3.0          9.211254   
4      CHEMBL4292264           MILADEMETAN        3.0          8.254145   
5      CHEMBL4297374  MILADEMETAN TOSYLATE        1.0               NaN   

   n_measurements  
0            25.0  
1             1.0  
2             3.0  
3             6.0  
4             1.0  
5             NaN  


In [14]:
actives.to_csv("chembl37_mdm2_actives.csv", index=False)
check.to_csv("chembl37_mdm2_known_drugs_check.csv", index=False)
print(len(actives), "active ذخیره شد")

3240 active ذخیره شد


In [15]:
cur.execute(sql_count, ("CHEMBL203",))

NameError: name 'sql_count' is not defined

In [16]:
count_sql = """
SELECT COUNT(*)
FROM activities AS a
JOIN assays AS s ON a.assay_id = s.assay_id
JOIN target_dictionary AS t ON s.tid = t.tid
WHERE t.chembl_id = ?
  AND a.standard_type IN ('IC50', 'Ki', 'Kd')
  AND a.standard_relation = '='
  AND a.standard_units = 'nM'
  AND s.assay_type = 'B'
"""
cur.execute(count_sql, ("CHEMBL203",))
print(cur.fetchone()[0])

19826


In [17]:
sql = """
SELECT
    a.activity_id,
    m.chembl_id AS molecule_chembl_id,
    cs.canonical_smiles,
    a.standard_type,
    a.standard_value,
    a.standard_units,
    a.standard_relation,
    s.chembl_id AS assay_chembl_id,
    s.assay_type,
    t.chembl_id AS target_chembl_id,
    a.pchembl_value,
    a.data_validity_comment,
    a.potential_duplicate
FROM activities AS a
JOIN assays AS s ON a.assay_id = s.assay_id
JOIN target_dictionary AS t ON s.tid = t.tid
JOIN molecule_dictionary AS m ON a.molregno = m.molregno
LEFT JOIN compound_structures AS cs ON a.molregno = cs.molregno
WHERE t.chembl_id = ?
  AND a.standard_type IN ('IC50', 'Ki', 'Kd')
  AND a.standard_relation = '='
  AND a.standard_units = 'nM'
  AND s.assay_type = 'B'
"""

raw_egfr = pd.read_sql_query(sql, con, params=("CHEMBL203",))
print(raw_egfr.shape)
print(raw_egfr.head(3))
raw_egfr.to_csv("chembl37_egfr_raw.csv", index=False)

(19826, 13)
   activity_id molecule_chembl_id  \
0        32260        CHEMBL68920   
1        32267        CHEMBL69960   
2        32680       CHEMBL137635   

                                    canonical_smiles standard_type  \
0  Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)...          IC50   
1  Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...          IC50   
2        CN(c1ccccc1)c1ncnc2ccc(N/N=N/Cc3ccccn3)cc12          IC50   

   standard_value standard_units standard_relation assay_chembl_id assay_type  \
0            41.0             nM                 =    CHEMBL674637          B   
1           170.0             nM                 =    CHEMBL674637          B   
2          9300.0             nM                 =    CHEMBL677833          B   

  target_chembl_id  pchembl_value data_validity_comment  potential_duplicate  
0        CHEMBL203           7.39                  None                    0  
1        CHEMBL203           6.77                  None                    

In [18]:
print(raw_egfr["standard_type"].value_counts())
print()
print("مولکول یکتا:", raw_egfr["molecule_chembl_id"].nunique())
print("بدون SMILES:", raw_egfr["canonical_smiles"].isna().sum())
print("standard_value <= 0:", (raw_egfr["standard_value"] <= 0).sum())
print()
print(raw_egfr["data_validity_comment"].value_counts(dropna=False))
print()
print(raw_egfr["potential_duplicate"].value_counts())

standard_type
IC50    18381
Kd        883
Ki        562
Name: count, dtype: int64

مولکول یکتا: 10833
بدون SMILES: 17
standard_value <= 0: 0

data_validity_comment
None                             19545
Outside typical range              223
Potential transcription error       58
Name: count, dtype: int64

potential_duplicate
0    17523
1     2303
Name: count, dtype: int64


In [19]:
clean_egfr = raw_egfr.copy()
print("شروع:", len(clean_egfr))

clean_egfr = clean_egfr.dropna(subset=["canonical_smiles"])
print("بعد از حذف بدون SMILES:", len(clean_egfr))

clean_egfr = clean_egfr[clean_egfr["data_validity_comment"].isna()]
print("بعد از حذف هشدار ChEMBL:", len(clean_egfr))

clean_egfr = clean_egfr[clean_egfr["potential_duplicate"] == 0]
print("بعد از حذف تکراری‌ها:", len(clean_egfr))

print("مولکول یکتا:", clean_egfr["molecule_chembl_id"].nunique())
print(clean_egfr["standard_type"].value_counts())

شروع: 19826
بعد از حذف بدون SMILES: 19809
بعد از حذف هشدار ChEMBL: 19528
بعد از حذف تکراری‌ها: 17299
مولکول یکتا: 10692
standard_type
IC50    16085
Kd        699
Ki        515
Name: count, dtype: int64


In [20]:
clean_egfr = clean_egfr.copy()
clean_egfr["pActivity"] = 9 - np.log10(clean_egfr["standard_value"])

print(clean_egfr["pActivity"].describe())
print()
print("pchembl خالی:", clean_egfr["pchembl_value"].isna().sum())
diff = (clean_egfr["pActivity"] - clean_egfr["pchembl_value"]).abs()
print("بیشترین اختلاف با pchembl:", diff.max())

count    17298.000000
mean         7.082898
std          1.335039
min          4.000000
25%          6.106460
50%          7.154902
75%          8.096910
max         11.000000
Name: pActivity, dtype: float64

pchembl خالی: 1
بیشترین اختلاف با pchembl: 0.005490200071283979


In [21]:
cols = ["activity_id", "molecule_chembl_id", "standard_type",
        "standard_value", "pchembl_value", "pActivity",
        "assay_chembl_id", "data_validity_comment"]

print("ردیف با pActivity خالی:")
print(clean_egfr[clean_egfr["pActivity"].isna()][cols])
print()
print("تعداد با pActivity >= 10.5:", (clean_egfr["pActivity"] >= 10.5).sum())
print(clean_egfr[clean_egfr["pActivity"] >= 10.5][cols].head(10))

ردیف با pActivity خالی:
     activity_id molecule_chembl_id standard_type  standard_value  \
944       740166        CHEMBL13983          IC50             NaN   

     pchembl_value  pActivity assay_chembl_id data_validity_comment  
944            NaN        NaN    CHEMBL677212                  None  

تعداد با pActivity >= 10.5: 23
       activity_id molecule_chembl_id standard_type  standard_value  \
1185        894024        CHEMBL29197          IC50           0.029   
1316       1021310        CHEMBL29197          IC50           0.025   
1345       1043415       CHEMBL176582          IC50           0.010   
1369       1054540       CHEMBL174426          IC50           0.025   
3506       3436088      CHEMBL1243316          IC50           0.030   
8826      18060758      CHEMBL4100860            Ki           0.030   
10693     22408226      CHEMBL3353410          IC50           0.020   
10698     22408231       CHEMBL180022          IC50           0.020   
10706     22408239       C

In [22]:
clean_egfr = clean_egfr.dropna(subset=["standard_value"])
print("بعد از حذف مقدار خالی:", len(clean_egfr))

mol_egfr = (
    clean_egfr.groupby("molecule_chembl_id")
              .agg(
                  canonical_smiles=("canonical_smiles", "first"),
                  pActivity_median=("pActivity", "median"),
                  pActivity_min=("pActivity", "min"),
                  pActivity_max=("pActivity", "max"),
                  n_measurements=("pActivity", "size"),
              )
              .reset_index()
)
mol_egfr["spread"] = mol_egfr["pActivity_max"] - mol_egfr["pActivity_min"]

print(mol_egfr.shape)
print(mol_egfr["n_measurements"].value_counts().head())
print("مولکول با پراکندگی > 1:", (mol_egfr["spread"] > 1).sum())
mol_egfr.to_csv("chembl37_egfr_molecules.csv", index=False)

بعد از حذف مقدار خالی: 17298
(10691, 7)
n_measurements
1    7345
2    2196
3     593
4     232
5     120
Name: count, dtype: int64
مولکول با پراکندگی > 1: 1407


In [23]:
actives_egfr = mol_egfr[mol_egfr["pActivity_median"] >= 7]
print("active:", len(actives_egfr), "از", len(mol_egfr))
print("  با فقط ۱ اندازه‌گیری:", (actives_egfr["n_measurements"] == 1).sum())
print("  با پراکندگی > 1:", (actives_egfr["spread"] > 1).sum())
actives_egfr.to_csv("chembl37_egfr_actives.csv", index=False)

active: 5499 از 10691
  با فقط ۱ اندازه‌گیری: 3544
  با پراکندگی > 1: 901


In [24]:
names_sql = """
SELECT chembl_id, pref_name, max_phase
FROM molecule_dictionary
WHERE UPPER(pref_name) LIKE '%ERLOTINIB%'
   OR UPPER(pref_name) LIKE '%GEFITINIB%'
   OR UPPER(pref_name) LIKE '%AFATINIB%'
   OR UPPER(pref_name) LIKE '%OSIMERTINIB%'
"""
known_egfr = pd.read_sql_query(names_sql, con)
known_egfr = known_egfr.rename(columns={"chembl_id": "molecule_chembl_id"})

check_egfr = known_egfr.merge(mol_egfr, on="molecule_chembl_id", how="left")
print(check_egfr[["molecule_chembl_id", "pref_name", "max_phase",
                  "pActivity_median", "n_measurements"]])
check_egfr.to_csv("chembl37_egfr_known_drugs_check.csv", index=False)

  molecule_chembl_id                pref_name  max_phase  pActivity_median  \
0          CHEMBL553                ERLOTINIB          4          7.545765   
1          CHEMBL939                GEFITINIB          4          7.767004   
2      CHEMBL1079742  ERLOTINIB HYDROCHLORIDE          4          7.489405   
3      CHEMBL1173655                 AFATINIB          4          8.958607   
4      CHEMBL2105712       AFATINIB DIMALEATE          4          9.349485   
5      CHEMBL3353410              OSIMERTINIB          4          7.720333   
6      CHEMBL3545063     OSIMERTINIB MESYLATE          4               NaN   

   n_measurements  
0           101.0  
1           117.0  
2             4.0  
3            66.0  
4             4.0  
5            95.0  
6             NaN  


In [25]:
ids = ["CHEMBL553", "CHEMBL939", "CHEMBL1173655", "CHEMBL3353410"]
print(mol_egfr[mol_egfr["molecule_chembl_id"].isin(ids)]
      [["molecule_chembl_id", "pActivity_min", "pActivity_median",
        "pActivity_max", "spread", "n_measurements"]])

      molecule_chembl_id  pActivity_min  pActivity_median  pActivity_max  \
219        CHEMBL1173655       5.853562          8.958607      10.214670   
2649       CHEMBL3353410       4.769551          7.720333      10.698970   
7283           CHEMBL553       4.036212          7.545765      10.537602   
10664          CHEMBL939       4.642065          7.767004      10.000000   

         spread  n_measurements  
219    4.361108              66  
2649   5.929419              95  
7283   6.501390             101  
10664  5.357935             117  


In [26]:
var_sql = """
SELECT s.chembl_id AS assay_chembl_id, s.variant_id
FROM assays AS s
JOIN target_dictionary AS t ON s.tid = t.tid
WHERE t.chembl_id = ?
"""
assay_var = pd.read_sql_query(var_sql, con, params=("CHEMBL203",))

tmp = clean_egfr.merge(assay_var, on="assay_chembl_id", how="left")
print("ردیف‌ها:", len(clean_egfr), "->", len(tmp))
tmp["has_variant"] = tmp["variant_id"].notna()
print(tmp["has_variant"].value_counts())
print()

d = tmp[tmp["molecule_chembl_id"].isin(ids)]
print(d.groupby(["molecule_chembl_id", "has_variant"])["pActivity"]
       .agg(["median", "min", "max", "size"]))

ردیف‌ها: 17298 -> 17298
has_variant
False    12792
True      4506
Name: count, dtype: int64

                                  median       min        max  size
molecule_chembl_id has_variant                                     
CHEMBL1173655      False        8.795880  5.853562  10.096910    27
                   True         9.013228  6.146302  10.214670    39
CHEMBL3353410      False        6.828537  5.547600   8.958607    28
                   True         8.229148  4.769551  10.698970    67
CHEMBL553          False        7.497573  5.206908  10.537602    67
                   True         7.922332  4.036212  10.229148    34
CHEMBL939          False        7.767004  4.642065  10.000000    67
                   True         7.776320  4.668775   9.769551    50


In [27]:
g = tmp[tmp["molecule_chembl_id"] == "CHEMBL939"].sort_values("pActivity")
cols = ["pActivity", "standard_type", "has_variant", "assay_chembl_id"]

print("ضعیف‌ترین ۵ مقدار:")
print(g[cols].head(5))
print()
print("قوی‌ترین ۵ مقدار:")
print(g[cols].tail(5))

ضعیف‌ترین ۵ مقدار:
       pActivity standard_type  has_variant assay_chembl_id
8296    4.642065          IC50        False   CHEMBL4179476
11326   4.668775          IC50         True   CHEMBL5320278
9884    4.809668          IC50        False   CHEMBL4827337
4907    4.854493          IC50         True   CHEMBL3137113
6015    4.895513          IC50         True   CHEMBL3743364

قوی‌ترین ۵ مقدار:
       pActivity standard_type  has_variant assay_chembl_id
12271   9.599980          IC50        False   CHEMBL5638814
4503    9.698970          IC50         True   CHEMBL2383987
7573    9.769551          IC50         True   CHEMBL3999846
7563    9.823909          IC50        False   CHEMBL3999845
4478   10.000000          IC50        False   CHEMBL2383984


In [28]:
def strict(df):
    return df[(df["pActivity_median"] >= 7)
              & (df["n_measurements"] >= 2)
              & (df["spread"] <= 1)]

strict_mdm2 = strict(mol_mdm2)
strict_egfr = strict(mol_egfr)

print("MDM2:", len(strict_mdm2), "از", len(actives))
print("EGFR:", len(strict_egfr), "از", len(actives_egfr))

strict_mdm2.to_csv("chembl37_mdm2_actives_strict.csv", index=False)
strict_egfr.to_csv("chembl37_egfr_actives_strict.csv", index=False)

MDM2: 441 از 3240
EGFR: 1054 از 5499


In [29]:
for name, strict_df, known in [("MDM2", strict_mdm2, check),
                               ("EGFR", strict_egfr, check_egfr)]:
    strict_ids = set(strict_df["molecule_chembl_id"])
    known = known.copy()
    known["in_strict"] = known["molecule_chembl_id"].isin(strict_ids)
    print(name)
    print(known[["pref_name", "pActivity_median",
                 "n_measurements", "in_strict"]])
    print()

MDM2
              pref_name  pActivity_median  n_measurements  in_strict
0              NUTLIN-3          7.045757            25.0      False
1              NUTLIN-2          6.853872             1.0      False
2           IDASANUTLIN          8.221849             3.0      False
3           NAVTEMADLIN          9.211254             6.0      False
4           MILADEMETAN          8.254145             1.0      False
5  MILADEMETAN TOSYLATE               NaN             NaN      False

EGFR
                 pref_name  pActivity_median  n_measurements  in_strict
0                ERLOTINIB          7.545765           101.0      False
1                GEFITINIB          7.767004           117.0      False
2  ERLOTINIB HYDROCHLORIDE          7.489405             4.0      False
3                 AFATINIB          8.958607            66.0      False
4       AFATINIB DIMALEATE          9.349485             4.0      False
5              OSIMERTINIB          7.720333            95.0      False
6 

In [30]:
ids_mdm2 = check["molecule_chembl_id"]
print(mol_mdm2[mol_mdm2["molecule_chembl_id"].isin(ids_mdm2)]
      [["molecule_chembl_id", "pActivity_min", "pActivity_median",
        "pActivity_max", "spread", "n_measurements"]])

     molecule_chembl_id  pActivity_min  pActivity_median  pActivity_max  \
99         CHEMBL191334       4.154902          7.045757       8.193820   
260       CHEMBL2402737       8.008774          8.221849       9.823909   
332       CHEMBL3125702       7.327902          9.211254      10.420216   
2170       CHEMBL407632       6.853872          6.853872       6.853872   
2455      CHEMBL4292264       8.254145          8.254145       8.254145   

        spread  n_measurements  
99    4.038918              25  
260   1.815135               3  
332   3.092314               6  
2170  0.000000               1  
2455  0.000000               1  
